# Constant Acceleration MPC Testing

In [ ]:
%matplotlib ipympl

import time
import functools
import jax
import jax.numpy as jnp
import numpy as np
import tqdm
import scipy.optimize as sci_opt

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.quartic_cost as quartic_cost

jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)


In [ ]:
# general setup

T = 10.0  # s
num_steps = int(T / const.dt)
n = 400  # horizon

acc_ref = jnp.array([1.0, 0.0, 0.0]) + const.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.0, 0.0, 0.0])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

In [ ]:
# cost setup

weights = opt.Weights(
    acc=jnp.array([1e1, 1e1, 1e0]),
    alpha_acc=jnp.array([4.0]),
)

# induces interesting periodic behavior...
# weights = opt.Weights(
#     acc=jnp.array([1e2, 1e2, 1e0]),
#     omega=jnp.array([1e2, 1e2, 1e2]),
#     alpha_acc=jnp.array([4.0]),
#     alpha_omega=jnp.array([4.0]),
# )

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_yaw,
    high=const.max_yaw,
)


In [ ]:
def table_sol_from_flat_control(state0: jax.Array, flat_control: jax.Array) -> utils.TableSol:
    control = opt.Control.from_flat(flat_control)
    state = control.get_state(state0)
    table_sol = utils.TableSol(
        x=jnp.array(jnp.column_stack(list(state))),
        u=jnp.array(jnp.column_stack(list(control))),
        stats=utils.TableStats(time=jnp.array(0.0), status=jnp.array(0), cost=jnp.array(0.0)),
    )
    return table_sol

In [ ]:
# train setup
acados_get_cost = functools.partial(
    opt.get_cost,
    weights=weights,
    acc_ref=acc_ref,
    omega_ref=omega_ref,
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
    # state0=state0,
)
    
def train_step(
    state0: jax.Array, last_control: jax.Array, **kwargs
) -> tuple[jax.Array, jax.Array, utils.TableSol, sci_opt.OptimizeResult, float]:
    cost, cost_jac, _ = acados_get_cost(state0=state0)
    t0 = time.time()
    opt_options = kwargs.get("opt_options", {"maxiter": 16, "maxls": 8})
    res = sci_opt.minimize(
        fun=cost,
        x0=np.array(last_control),
        method="L-BFGS-B",
        jac=cost_jac,
        options=opt_options,
    )
    t1 = time.time()
    t_tot = t1 - t0
    control = opt.Control.from_flat(jnp.array(res.x))
    state = control.get_state(state0)
    table_sol = utils.TableSol(
        x=jnp.column_stack(list(state)),
        u=jnp.column_stack(list(control)),
        stats=utils.TableStats(
            time=jnp.squeeze(t_tot),
            status=jnp.squeeze(res.status),
            cost=jnp.squeeze(res.fun),
        )
    )
    return state[1].flatten(), control.flatten(), table_sol, res, t_tot


In [ ]:
# run setup
state0 = jnp.zeros(12, dtype=float)
_, last_control, _, _, _ = train_step(state0, jnp.zeros(6 * n, dtype=float))
last_control.block_until_ready()

# with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
#     _, last_control, _, _, _ = train_step(state0, jnp.zeros(6 * n, dtype=float))
#     last_control.block_until_ready()

times = []
sol_list = []
res_list = []

In [ ]:
# run
for _ in tqdm.tqdm(range(num_steps)):
# for _ in range(num_steps):
    state0, last_control, sol, res, t_tot = train_step(state0, last_control, opt_options={"maxiter": 2, "maxls": 4})
    sol_list.append(sol)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
def max_abs(arr):
    return jnp.max(jnp.abs(arr))

def max_abs_arg(arr):
    return jnp.argmax(jnp.abs(arr))

arr = opt.Control.from_flat(sol_list[-1].u.flatten()).z
max_abs(arr), max_abs_arg(arr)  # , arr

In [ ]:
plt.close("all")

In [ ]:
sol_list_end = []
extra_steps = 0

sol_list_end = viz.split_tablesol(sol_list[-1])
extra_steps = sol_list[-1].u.shape[0] - 1

In [ ]:
acados_human_fig = viz.plot_human_trajectory(
    trajectory=sol_list + sol_list_end,
    references={
        "xyz-acceleration": np.tile(
            A=acc_ref[0], reps=(len(sol_list) + len(sol_list_end), 1)
        ),
        "angular-velocity": np.tile(
            A=omega_ref[0], reps=(len(sol_list) + len(sol_list_end), 1)
        ),
    },
)

In [ ]:
acados_table_fig = viz.plot_cartesian_table_trajectory(
    trajectory=sol_list + sol_list_end,
)

In [ ]:
acados_actuator_fig = viz.plot_actuator_trajectory(
    trajectory=sol_list + sol_list_end,
)

In [ ]:
# u = np.asarray(sol_list[len(sol_list) // 2].u)  # shape: (horizon, 6)
u = np.asarray(sol_list[-1].u)  # shape: (horizon, 6)
t = np.arange(u.shape[0]) * const.dt

fig, axs = plt.subplots(6, 1, figsize=(10, 8), sharex=True)
for i, ax in enumerate(axs):
    ax.plot(t, u[:, i], '-o', markersize=3, linewidth=1)
    ax.set_ylabel(f'u[{i}]')
    ax.grid(True)

axs[-1].set_xlabel('time (s)')
fig.suptitle('Control horizon — last MPC solution')
fig.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.show()

In [ ]:
u = np.asarray(sol_list[len(sol_list) // 2].u)  # shape: (horizon, 6)
# u = np.asarray(sol_list[-1].u)  # shape: (horizon, 6)
t = np.arange(u.shape[0]) * const.dt

fig, axs = plt.subplots(6, 1, figsize=(10, 8), sharex=True)
for i, ax in enumerate(axs):
    ax.plot(t, u[:, i], '-o', markersize=3, linewidth=1)
    ax.set_ylabel(f'u[{i}]')
    ax.grid(True)

axs[-1].set_xlabel('time (s)')
fig.suptitle('Control horizon — last MPC solution')
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
cost, cost_jac, _ = acados_get_cost(state0=state0)
cost(u), cost(np.zeros_like(u))

In [ ]:
# saving the animation takes a long time, so it should be at the end of the notebook

anim_fig = viz.animate_trajectory(
    trajectory=sol_list + sol_list_end,
    sim_rate=1.0,
    fps=30,
)
anim_fig[0].save("data/const_acc.mp4", writer="ffmpeg", dpi=250)
plt.close(anim_fig[1])